# Task 1 CRNN Training on Colab

This notebook runs the `task1_crnn` training and evaluation pipeline on Google Colab for either `T4` or `H100`.

Important:
- Use a fresh `wikiart.zip` in Google Drive.
- If local extraction fails, the notebook will fall back to zip-backed loading.
- `H100` should use the `H100` profile. `T4` should use the `T4` profile.

## Expected Drive Layout

```text
MyDrive/
  GSoC/
    ArtGAN/
      colab/
      task1_crnn/
      WikiArt Dataset/
      TASK1_CONV_RECURRENT.md
      COLAB_TASK1.md
    wikiart.zip
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

MYDRIVE = Path('/content/drive/MyDrive')

REPO_CANDIDATES = [
    MYDRIVE / 'GSoC' / 'ArtGAN',
    MYDRIVE / 'ArtGAN',
]
DATA_ZIP_CANDIDATES = [
    MYDRIVE / 'GSoC' / 'wikiart.zip',
    MYDRIVE / 'wikiart.zip',
    MYDRIVE / 'GSoC' / 'ArtGAN' / 'wikiart.zip',
]

DRIVE_REPO_DIR = next((path for path in REPO_CANDIDATES if path.exists()), REPO_CANDIDATES[0])
DRIVE_DATA_ZIP = next((path for path in DATA_ZIP_CANDIDATES if path.exists()), DATA_ZIP_CANDIDATES[0])
PROJECT_ROOT = DRIVE_REPO_DIR.parent

WORKDIR = Path('/content/ArtGAN')
LOCAL_DATA_ZIP = Path('/content/wikiart.zip')
OUTPUTS_DIR = PROJECT_ROOT / 'ArtGAN_outputs' / 'task1_crnn'

print('REPO_CANDIDATES =', REPO_CANDIDATES)
print('DATA_ZIP_CANDIDATES =', DATA_ZIP_CANDIDATES)
print('DRIVE_REPO_DIR =', DRIVE_REPO_DIR)
print('DRIVE_DATA_ZIP =', DRIVE_DATA_ZIP)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('WORKDIR =', WORKDIR)
print('LOCAL_DATA_ZIP =', LOCAL_DATA_ZIP)
print('OUTPUTS_DIR =', OUTPUTS_DIR)


In [ ]:
import os
import shutil
from pathlib import Path

if not DRIVE_REPO_DIR.exists():
    raise FileNotFoundError('Missing repo folder. Checked: ' + ', '.join(str(path) for path in REPO_CANDIDATES))
if not DRIVE_DATA_ZIP.exists():
    raise FileNotFoundError('Missing dataset zip. Checked: ' + ', '.join(str(path) for path in DATA_ZIP_CANDIDATES))

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
shutil.copytree(DRIVE_REPO_DIR, WORKDIR, dirs_exist_ok=True)
shutil.copy2(DRIVE_DATA_ZIP, LOCAL_DATA_ZIP)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)

print('Copied repo to', WORKDIR)
print('Copied dataset zip to', LOCAL_DATA_ZIP)
print('Outputs will be saved under', OUTPUTS_DIR)
print('Current working directory:', Path.cwd())


In [ ]:
%pip install -q scikit-learn pillow tqdm


In [ ]:
!python -m task1_crnn.audit --dataset-dir "WikiArt Dataset" --output-json outputs/task1_crnn/audit.json


## Profile Selection

Set `HARDWARE_PROFILE` to either `"T4"` or `"H100"`.

- `T4`: safer defaults, zip-backed loading, smaller batch sizes
- `H100`: larger batch sizes and local extraction when possible

`SHOW_BATCH_PROGRESS = True` tries to show batch progress. If Colab output becomes noisy or unstable, set it to `False`.

In [ ]:
HARDWARE_PROFILE = "H100"  # change to "T4" when needed
SHOW_BATCH_PROGRESS = True
print("HARDWARE_PROFILE =", HARDWARE_PROFILE)
print("SHOW_BATCH_PROGRESS =", SHOW_BATCH_PROGRESS)


In [ ]:
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

LOCAL_EXTRACT_DIR = Path("/content/wikiart_unzipped")
LOCAL_IMAGE_ROOT = LOCAL_EXTRACT_DIR / "wikiart"
free_gb = shutil.disk_usage("/content").free / (1024 ** 3)
USE_LOCAL_IMAGE_ROOT = False

if HARDWARE_PROFILE.upper() == "H100" and free_gb >= 35:
    if LOCAL_IMAGE_ROOT.exists():
        USE_LOCAL_IMAGE_ROOT = True
    else:
        try:
            subprocess.run(["unzip", "-q", "-o", "/content/wikiart.zip", "-d", str(LOCAL_EXTRACT_DIR)], check=True)
            USE_LOCAL_IMAGE_ROOT = LOCAL_IMAGE_ROOT.exists()
        except subprocess.CalledProcessError as error:
            USE_LOCAL_IMAGE_ROOT = False
            print("Extraction failed; falling back to zip-backed loading.")
            print("This usually means wikiart.zip has at least one corrupted entry.")
            print("unzip exit code =", error.returncode)
elif HARDWARE_PROFILE.upper() == "H100":
    print(f"Skipping extraction: only {free_gb:.1f} GB free in /content")

if HARDWARE_PROFILE.upper() == "H100":
    TRAIN_EPOCHS = 16
    TRAIN_BATCH_SIZE = 32
    TRAIN_NUM_WORKERS = 4 if USE_LOCAL_IMAGE_ROOT else 0
    EVAL_BATCH_SIZE = 64 if USE_LOCAL_IMAGE_ROOT else 16
    EVAL_NUM_WORKERS = 4 if USE_LOCAL_IMAGE_ROOT else 0
else:
    TRAIN_EPOCHS = 12
    TRAIN_BATCH_SIZE = 8
    TRAIN_NUM_WORKERS = 0
    EVAL_BATCH_SIZE = 16
    EVAL_NUM_WORKERS = 0

resume_path = Path("outputs/task1_crnn/last.pt")
PROGRESS_ARGS = [] if SHOW_BATCH_PROGRESS else ["--disable-progress"]
IMAGE_ROOT_ARGS = ["--image-root", str(LOCAL_IMAGE_ROOT)] if USE_LOCAL_IMAGE_ROOT else []
RESUME_ARGS = ["--resume-from", str(resume_path)] if resume_path.exists() else []

print("free_gb =", round(free_gb, 1))
print("USE_LOCAL_IMAGE_ROOT =", USE_LOCAL_IMAGE_ROOT)
print("LOCAL_IMAGE_ROOT =", LOCAL_IMAGE_ROOT if USE_LOCAL_IMAGE_ROOT else "zip-backed loader")
print("TRAIN_EPOCHS =", TRAIN_EPOCHS)
print("TRAIN_BATCH_SIZE =", TRAIN_BATCH_SIZE)
print("TRAIN_NUM_WORKERS =", TRAIN_NUM_WORKERS)
print("EVAL_BATCH_SIZE =", EVAL_BATCH_SIZE)
print("EVAL_NUM_WORKERS =", EVAL_NUM_WORKERS)
print("RESUME =", resume_path.exists())


## Smoke Test

Run this once before full training. It checks the pipeline end-to-end with a very small run.

In [ ]:
SMOKE_CMD = [
    sys.executable, "-u", "-m", "task1_crnn.train",
    "--dataset-dir", "WikiArt Dataset",
    "--archive-path", "/content/wikiart.zip",
    *IMAGE_ROOT_ARGS,
    "--output-dir", "outputs/task1_crnn_smoke",
    "--epochs", "1",
    "--batch-size", "4",
    "--num-workers", "0",
    "--limit-train-batches", "10",
    "--limit-val-batches", "10",
    "--image-size", "224",
    "--crop-size", "224",
    *PROGRESS_ARGS,
]
print("Running:", shlex.join(SMOKE_CMD))
get_ipython().system(shlex.join(SMOKE_CMD))


## Full Training

Run this after the smoke test passes.

- `T4` uses zip-backed loading
- `H100` uses local extracted images when available
- training resumes from `outputs/task1_crnn/last.pt` automatically if present

In [ ]:
TRAIN_CMD = [
    sys.executable, "-u", "-m", "task1_crnn.train",
    "--dataset-dir", "WikiArt Dataset",
    "--archive-path", "/content/wikiart.zip",
    *IMAGE_ROOT_ARGS,
    "--output-dir", "outputs/task1_crnn",
    "--epochs", str(TRAIN_EPOCHS),
    "--batch-size", str(TRAIN_BATCH_SIZE),
    "--num-workers", str(TRAIN_NUM_WORKERS),
    "--pretrained-backbone",
    "--grad-clip-norm", "1.0",
    *PROGRESS_ARGS,
    *RESUME_ARGS,
]
print("Running:", shlex.join(TRAIN_CMD))
get_ipython().system(shlex.join(TRAIN_CMD))


## Evaluation and Outliers

This uses the same image source as the training profile and writes metrics plus outlier reports into `outputs/task1_crnn/eval`.

In [ ]:
EVAL_CMD = [
    sys.executable, "-u", "-m", "task1_crnn.evaluate",
    "--checkpoint", "outputs/task1_crnn/best.pt",
    "--dataset-dir", "WikiArt Dataset",
    "--archive-path", "/content/wikiart.zip",
    *IMAGE_ROOT_ARGS,
    "--output-dir", "outputs/task1_crnn/eval",
    "--batch-size", str(EVAL_BATCH_SIZE),
    "--num-workers", str(EVAL_NUM_WORKERS),
    "--top-outliers", "25",
]
print("Running:", shlex.join(EVAL_CMD))
get_ipython().system(shlex.join(EVAL_CMD))


In [ ]:
import shutil

target = OUTPUTS_DIR
target.mkdir(parents=True, exist_ok=True)
if (target / "task1_crnn").exists():
    shutil.rmtree(target / "task1_crnn")
shutil.copytree(WORKDIR / "outputs" / "task1_crnn", target / "task1_crnn", dirs_exist_ok=True)
print("Saved outputs to", target / "task1_crnn")


## Notes

- If `H100` extraction fails, replace `wikiart.zip` with a fresh clean copy and rerun the profile config cell.
- If Colab disconnects, rerun the setup cells and the full training cell. The notebook will resume from `outputs/task1_crnn/last.pt` when it exists.
- If batch progress makes the notebook unstable, set `SHOW_BATCH_PROGRESS = False` in the profile cell.